# Attention Conv-LSTM — Travel Time Model

A second sequence model for the same per-stop travel-time regression task as
`train_lstm.ipynb`, but with a different encoder:

**Embeddings → Causal Conv1D stack → LSTM → Causal Multi-Head Self-Attention (transformer block) → MLP head**

Rationale for each piece vs. the plain LSTM baseline:
- **Causal Conv1D** ahead of the recurrent layer lets the model learn local, translation-invariant
  patterns across a few neighboring stops (e.g. a cluster of closely-spaced stops, a short signal-heavy
  block) before anything is folded into recurrent state.
- **LSTM** still carries the sequential/positional state — it knows the order of stops and can carry a
  running "how are we doing so far" signal (accumulated delay, etc).
- **Causal self-attention** on top of the LSTM outputs gives every stop a *direct* path to attend to any
  earlier stop in the same trip (e.g. "we already lost 90s at stop 4" propagating straight to stop 40),
  instead of relying purely on what survives being pushed through the LSTM's recurrent bottleneck.
  It's masked so a stop can never attend to a future stop (no leakage), and padding is masked too.
- A residual + LayerNorm + feed-forward block (the standard transformer sandwich) follows the attention,
  which usually makes attention easier to train jointly with the recurrent part.

Everything upstream of the model (data loading, categorical encoding, numeric normalization, sequence
building, Dataset/collate, masked loss functions) mirrors `train_lstm.ipynb` exactly, so the two models
are trained/evaluated on identical splits and are directly comparable.


In [2]:
import numpy as np
import polars as pl
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit

# =====================================================================
# 0. Config — mirrors train_lstm.ipynb's feature split
# =====================================================================
DATA_PATH = "raw/processed_gtfs/baseline_dataset.parquet"
TARGET = "travel_time"
SEQ_ID_COLS = ["trip_id", "start_date"]   # what defines one sequence
ORDER_COL = "stop_sequence"

CATEGORICAL = ["route_id", "direction_id", "shape_id", "service_id", 'is_raining', 'is_snowing', 'is_fog', "is_weekend",
    "is_federal_holiday",
    "is_school_day",
    "has_major_event", "is_peak", 'weathercode',]
NUMERIC = [
    "stop_sequence", "trip_progress",
    "hour", "weekday", "month",
    "scheduled_arrival", "scheduled_departure", "scheduled_segment_time",
    "stop_lat", "stop_lon", "latitude", "longitude", "bearing",
    'temperature_c', 'precipitation_mm', 'snowfall_cm', 'windspeed_kmh',
    'segment_length',
    'scheduled_segment_speed_mps',
    'upstream_delay_seconds',
    'speed_mps',
    'headway_seconds',
    "ridership",
    "transfers",
]

EMB_DIM_CAP = 150     # max embedding dim per categorical column

# --- model-specific hyperparameters (Conv-Attention-LSTM) ---
HIDDEN_SIZE = 256        # LSTM hidden size / attention model dim
NUM_LAYERS = 2           # LSTM layers
CONV_CHANNELS = 256      # channels produced by the causal conv stack
CONV_KERNEL = 3          # causal conv kernel size
N_HEADS = 4              # attention heads (HIDDEN_SIZE must be divisible by N_HEADS)
FFN_MULT = 4             # feed-forward expansion after attention
DROPOUT = 0.1

BATCH_SIZE = 64
LR = 1e-3
MAX_EPOCHS = 100
PATIENCE = 12            # early stopping
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
np.random.seed(42)


In [3]:
df = pl.read_parquet(DATA_PATH)
print(f"Loaded {df.shape}")

df = df.with_columns(
    (pl.col("trip_id") + "_" + pl.col("start_date").cast(pl.Utf8)).alias("_seq_id")
)

groups = df["_seq_id"].to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=groups))
train_df = df[train_idx]
test_df = df[test_idx]
print(f"train rows: {train_df.height:,} | test rows: {test_df.height:,}")
print(f"train trips: {train_df['_seq_id'].n_unique():,} | test trips: {test_df['_seq_id'].n_unique():,}")


Loaded (5596768, 40)
train rows: 4,478,560 | test rows: 1,118,208
train trips: 96,308 | test trips: 24,077


In [4]:
# =====================================================================
# 2. Encode categoricals — fit on TRAIN only, reserve 0 for padding/unknown
# =====================================================================
cat_maps = {}
cat_cardinalities = []
for col in CATEGORICAL:
    uniques = train_df[col].unique().sort().to_list()
    cat_maps[col] = {v: i + 1 for i, v in enumerate(uniques)}  # 0 = pad/unknown
    cat_cardinalities.append(len(uniques) + 1)  # +1 for the unknown/pad bucket


def encode_categoricals(d: pl.DataFrame) -> pl.DataFrame:
    exprs = []
    for col in CATEGORICAL:
        mapping = cat_maps[col]
        exprs.append(
            pl.col(col).cast(pl.Utf8).replace_strict(mapping, default=0).cast(pl.Int64).alias(f"_{col}_code")
        )
    return d.with_columns(exprs)


train_df = encode_categoricals(train_df)
test_df = encode_categoricals(test_df)
CAT_CODE_COLS = [f"_{c}_code" for c in CATEGORICAL]


In [5]:
# =====================================================================
# 3. Normalize numerics — fit mean/std on TRAIN only
# =====================================================================
num_means = {c: train_df[c].mean() for c in NUMERIC}
num_stds = {c: (train_df[c].std() or 1.0) for c in NUMERIC}
num_stds = {c: (s if s and s > 1e-8 else 1.0) for c, s in num_stds.items()}


def normalize_numeric(d: pl.DataFrame) -> pl.DataFrame:
    return d.with_columns([
        ((pl.col(c) - num_means[c]) / num_stds[c]).alias(c) for c in NUMERIC
    ])


train_df = normalize_numeric(train_df)
test_df = normalize_numeric(test_df)


In [6]:
# =====================================================================
# 4. Build one sequence (stops sorted by stop_sequence) per trip
# =====================================================================
def build_sequences(d: pl.DataFrame) -> list[dict]:
    d = d.sort(SEQ_ID_COLS + [ORDER_COL])
    sequences = []
    for _, group in d.group_by("_seq_id", maintain_order=True):
        cat = group.select(CAT_CODE_COLS).to_numpy()
        num = group.select(NUMERIC).to_numpy().astype(np.float32)
        target = group[TARGET].to_numpy().astype(np.float32)
        sequences.append({"cat": cat, "num": num, "target": target, "length": len(target)})
    return sequences


print("Building train sequences...")
train_sequences = build_sequences(train_df)
print("Building test sequences...")
test_sequences = build_sequences(test_df)
print(f"train sequences: {len(train_sequences):,} | test sequences: {len(test_sequences):,}")


Building train sequences...
Building test sequences...
train sequences: 96,308 | test sequences: 24,077


In [7]:
# =====================================================================
# 5. Dataset / collate — pad each batch to its own max length
# =====================================================================
class TripSequenceDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        s = self.sequences[idx]
        return (
            torch.tensor(s["cat"], dtype=torch.long),
            torch.tensor(s["num"], dtype=torch.float32),
            torch.tensor(s["target"], dtype=torch.float32),
            s["length"],
        )


def collate(batch):
    cats, nums, targets, lengths = zip(*batch)
    lengths = torch.tensor(lengths, dtype=torch.long)
    cats_padded = pad_sequence(cats, batch_first=True, padding_value=0)
    nums_padded = pad_sequence(nums, batch_first=True, padding_value=0.0)
    targets_padded = pad_sequence(targets, batch_first=True, padding_value=0.0)
    return cats_padded, nums_padded, targets_padded, lengths


train_loader = DataLoader(TripSequenceDataset(train_sequences), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
test_loader = DataLoader(TripSequenceDataset(test_sequences), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)


## 6. Model — Conv-Attention-LSTM

```
categorical codes ─► embeddings ─┐
                                  ├─ concat ─► causal Conv1D x2 (ReLU) ─► LayerNorm
numeric features ─────────────────┘                                        │
                                                                             ▼
                                                                       LSTM (2 layers)
                                                                             │
                                                        causal multi-head self-attention
                                                        (query=key=value=LSTM output,
                                                         upper-triangular mask + padding mask)
                                                                             │
                                                          residual + LayerNorm + FFN + residual
                                                                             │
                                                                        MLP head ─► per-stop travel time
```

Both the causal convolutions and the attention mask enforce that stop *t* only ever sees information
from stops `<= t` — no leakage of future stops into a prediction, matching the LSTM baseline's causal
behaviour but adding a direct (non-recurrent) long-range path via attention.


In [8]:
class ConvAttnLSTMTravelTime(nn.Module):
    def __init__(self, cat_cardinalities, n_numeric, hidden_size, num_layers,
                 conv_channels, conv_kernel, n_heads, ffn_mult, dropout):
        super().__init__()
        assert hidden_size % n_heads == 0, "hidden_size must be divisible by n_heads"

        # Fast.ai-style dynamic embedding sizing rule (same as baseline)
        emb_dims = [min(EMB_DIM_CAP, max(1, int(1.6 * (c ** 0.56)))) for c in cat_cardinalities]
        self.embeddings = nn.ModuleList([
            nn.Embedding(c, d, padding_idx=0) for c, d in zip(cat_cardinalities, emb_dims)
        ])
        in_dim = sum(emb_dims) + n_numeric

        # --- causal Conv1D stack ---
        # padding=(k-1) on the left only, achieved by symmetric padding + right-trim,
        # so output at position t depends only on inputs at positions <= t.
        self.conv_kernel = conv_kernel
        self.conv1 = nn.Conv1d(in_dim, conv_channels, conv_kernel, padding=conv_kernel - 1)
        self.conv2 = nn.Conv1d(conv_channels, conv_channels, conv_kernel, padding=conv_kernel - 1)
        self.conv_act = nn.ReLU()
        self.conv_norm = nn.LayerNorm(conv_channels)
        self.conv_dropout = nn.Dropout(dropout)

        # --- recurrent layer ---
        self.lstm = nn.LSTM(
            conv_channels, hidden_size, num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        # --- causal self-attention (transformer) block ---
        self.attn = nn.MultiheadAttention(hidden_size, n_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * ffn_mult),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size * ffn_mult, hidden_size),
        )
        self.ffn_norm = nn.LayerNorm(hidden_size)

        # --- output head ---
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),
        )

    def _causal_conv(self, conv, x):
        # x: (B, C_in, T) -> (B, C_out, T), trimming the extra right-padding so
        # position t only sees inputs at positions <= t.
        out = conv(x)
        trim = conv.kernel_size[0] - 1
        return out[:, :, :-trim] if trim > 0 else out

    def forward(self, cat, num, lengths):
        B, T, _ = num.shape
        emb = [e(cat[:, :, i]) for i, e in enumerate(self.embeddings)]
        x = torch.cat(emb + [num], dim=-1)                   # (B, T, in_dim)

        xc = x.transpose(1, 2)                                # (B, in_dim, T)
        xc = self.conv_act(self._causal_conv(self.conv1, xc))
        xc = self.conv_act(self._causal_conv(self.conv2, xc))
        xc = xc.transpose(1, 2)                               # (B, T, conv_channels)
        xc = self.conv_dropout(self.conv_norm(xc))

        packed = pack_padded_sequence(xc, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        lstm_out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=T)

        # causal mask: True = disallowed. Position i may attend to positions <= i only.
        causal_mask = torch.triu(torch.ones(T, T, device=num.device, dtype=torch.bool), diagonal=1)
        # padding mask: True = ignore (position is beyond this sequence's real length)
        key_padding_mask = torch.arange(T, device=num.device).unsqueeze(0) >= lengths.unsqueeze(1).to(num.device)

        attn_out, _ = self.attn(
            lstm_out, lstm_out, lstm_out,
            attn_mask=causal_mask,
            key_padding_mask=key_padding_mask,
            need_weights=False,
        )
        x1 = self.attn_norm(lstm_out + attn_out)

        ffn_out = self.ffn(x1)
        x2 = self.ffn_norm(x1 + ffn_out)

        return self.head(x2).squeeze(-1)


def masked_mae(pred: torch.Tensor, target: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    mask = (torch.arange(pred.size(1), device=pred.device).unsqueeze(0) < lengths.unsqueeze(1).to(pred.device))
    return (torch.abs(pred - target) * mask).sum() / mask.sum().clamp(min=1)


def masked_mse(pred: torch.Tensor, target: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    mask = (torch.arange(pred.size(1), device=pred.device).unsqueeze(0) < lengths.unsqueeze(1).to(pred.device))
    return (((pred - target) ** 2) * mask).sum() / mask.sum().clamp(min=1)


model = ConvAttnLSTMTravelTime(
    cat_cardinalities, len(NUMERIC), HIDDEN_SIZE, NUM_LAYERS,
    CONV_CHANNELS, CONV_KERNEL, N_HEADS, FFN_MULT, DROPOUT,
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=3e-3,
    epochs=MAX_EPOCHS,
    steps_per_epoch=len(train_loader),
)
print(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {n_params:,}")


ConvAttnLSTMTravelTime(
  (embeddings): ModuleList(
    (0): Embedding(6, 4, padding_idx=0)
    (1): Embedding(3, 2, padding_idx=0)
    (2): Embedding(79, 18, padding_idx=0)
    (3): Embedding(49, 14, padding_idx=0)
    (4-5): 2 x Embedding(3, 2, padding_idx=0)
    (6): Embedding(2, 2, padding_idx=0)
    (7-11): 5 x Embedding(3, 2, padding_idx=0)
    (12): Embedding(14, 7, padding_idx=0)
  )
  (conv1): Conv1d(85, 256, kernel_size=(3,), stride=(1,), padding=(2,))
  (conv2): Conv1d(256, 256, kernel_size=(3,), stride=(1,), padding=(2,))
  (conv_act): ReLU()
  (conv_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
  (conv_dropout): Dropout(p=0.1, inplace=False)
  (lstm): LSTM(256, 256, num_layers=2, batch_first=True, dropout=0.1)
  (attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
  )
  (attn_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
  (ffn): Sequential(
    (0): Lin

In [9]:
# =====================================================================
# 7. Train loop with early stopping on validation (test) MAE
# =====================================================================
def run_epoch(loader, train: bool) -> float:
    model.train() if train else model.eval()
    total_mae, total_count = 0.0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for cat, num, target, lengths in loader:
            cat, num, target = cat.to(DEVICE), num.to(DEVICE), target.to(DEVICE)
            pred = model(cat, num, lengths)
            loss = masked_mse(pred, target, lengths)
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
                scheduler.step()  # per-batch LR update (OneCycleLR)
            mae = masked_mae(pred, target, lengths)
            n = lengths.sum().item()
            total_mae += mae.item() * n
            total_count += n
    return total_mae / total_count


best_val_mae = float("inf")
epochs_no_improve = 0
best_state = None

for epoch in range(1, MAX_EPOCHS + 1):
    train_mae = run_epoch(train_loader, train=True)
    val_mae = run_epoch(test_loader, train=False)
    print(f"epoch {epoch:3d} | train MAE {train_mae:7.2f}s | val MAE {val_mae:7.2f}s")

    if val_mae < best_val_mae - 1e-3:
        best_val_mae = val_mae
        epochs_no_improve = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (best val MAE {best_val_mae:.2f}s)")
            break

model.load_state_dict(best_state)


epoch   1 | train MAE   44.19s | val MAE   32.81s
epoch   2 | train MAE   32.40s | val MAE   31.35s
epoch   3 | train MAE   31.67s | val MAE   30.73s
epoch   4 | train MAE   31.37s | val MAE   30.61s
epoch   5 | train MAE   31.17s | val MAE   30.76s
epoch   6 | train MAE   31.00s | val MAE   30.82s
epoch   7 | train MAE   30.90s | val MAE   30.61s
epoch   8 | train MAE   30.77s | val MAE   30.84s
epoch   9 | train MAE   30.69s | val MAE   30.76s
epoch  10 | train MAE   30.58s | val MAE   30.78s
epoch  11 | train MAE   30.51s | val MAE   30.19s
epoch  12 | train MAE   30.46s | val MAE   30.03s
epoch  13 | train MAE   30.38s | val MAE   29.90s
epoch  14 | train MAE   30.35s | val MAE   29.97s
epoch  15 | train MAE   30.32s | val MAE   30.74s
epoch  16 | train MAE   30.30s | val MAE   30.17s
epoch  17 | train MAE   30.25s | val MAE   29.84s
epoch  18 | train MAE   30.24s | val MAE   30.43s
epoch  19 | train MAE   30.25s | val MAE   30.13s
epoch  20 | train MAE   30.26s | val MAE   29.98s


<All keys matched successfully>

In [10]:
# =====================================================================
# 8. Final evaluation — same metrics as train_lstm.ipynb / LightGBM model
#    so all three are directly comparable
# =====================================================================
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for cat, num, target, lengths in test_loader:
        cat, num = cat.to(DEVICE), num.to(DEVICE)
        pred = model(cat, num, lengths).cpu()
        for i, length in enumerate(lengths):
            all_preds.append(pred[i, :length].numpy())
            all_targets.append(target[i, :length].numpy())

preds = np.concatenate(all_preds)
targets = np.concatenate(all_targets)
resid = preds - targets

mae = np.abs(resid).mean()
rmse = np.sqrt((resid ** 2).mean())
median_ae = np.median(np.abs(resid))
bias = resid.mean()
ss_res = (resid ** 2).sum()
ss_tot = ((targets - targets.mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot

print(f"\nMAE:       {mae:.1f} sec")
print(f"Median AE: {median_ae:.1f} sec")
print(f"RMSE:      {rmse:.1f} sec")
print(f"Bias:      {bias:+.1f} sec")
print(f"R2:        {r2:.4f}")
for thresh in (30, 60, 120):
    print(f"within {thresh}s: {(np.abs(resid) <= thresh).mean():.1%}")

import os
os.makedirs("models", exist_ok=True)
torch.save(
    {
        "model_state": model.state_dict(),
        "cat_maps": cat_maps,
        "num_means": num_means,
        "num_stds": num_stds,
        "config": {
            "HIDDEN_SIZE": HIDDEN_SIZE,
            "NUM_LAYERS": NUM_LAYERS,
            "CONV_CHANNE    LS": CONV_CHANNELS,
            "CONV_KERNEL": CONV_KERNEL,
            "N_HEADS": N_HEADS,
            "FFN_MULT": FFN_MULT,
            "DROPOUT": DROPOUT,
            "NUMERIC": NUMERIC,
            "CATEGORICAL": CATEGORICAL,
        },
    },
    "models/conv_attn_lstm.pt",
)
print("\nmodel saved to models/conv_attn_lstm.pt")



MAE:       29.8 sec
Median AE: 20.0 sec
RMSE:      54.2 sec
Bias:      -3.3 sec
R2:        0.5100
within 30s: 65.9%
within 60s: 89.9%
within 120s: 97.8%

model saved to models/conv_attn_lstm.pt
